# Create files mapping FACS well and plate to basic allele metadata

This script takes cherry picking and allele info files as input and produces a table relating FACS well and plate IDs to allele IDs and symbols. 

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os

In [2]:
dualipa_inputs = "../1_inputs"
dualipa_outputs = "../3_outputs"
allele_collection_inputs = "../../../1_allele_collection/1_inputs/seq_confirmation_results"
allele_collection_outputs = "../../../1_allele_collection/3_outputs"

## 1. Loading the pdest layout data frame

Vidal lab ran experiment once and analyzed the data. Some alleles didn't pass the qa/qc thresholds or got no data. Batch5 is is a re-run for all alleles that failed in the first set of batches. Here, we process everything together from scratch. An allele should always be compared to the wt on the same batch and plate. The summary stats can then be compared across batches. 

In [3]:
# Read in cherry picking files for both the initial batches and the second round
cp_col = ['orf_id', 'mut_id', 'node_type','dest_pla_id', 'dest_pla', 'dest_well']

pdest_layout_initial_batches_df = pd.read_csv(f'{dualipa_inputs}/cherry_picking_files/summary_CP_VUSDUALXP.csv', index_col=0)[cp_col]
pdest_layout_b5_df = pd.read_csv(f'{dualipa_inputs}/cherry_picking_files/summary_CP_VUSDUALXP_batch5.csv', index_col=0)[cp_col]

pdest_layout_df = pd.concat([pdest_layout_b5_df, pdest_layout_initial_batches_df])
pdest_layout_df = pl.DataFrame(pdest_layout_df)
pdest_layout_df = pdest_layout_df.with_columns(
    pl.col("orf_id").cast(pl.Int64).alias("orf_id"),
    pl.col("mut_id").cast(pl.Int64).alias("mut_id")
)
pdest_layout_df
### This doesn't have well A01

orf_id,mut_id,node_type,dest_pla_id,dest_pla,dest_well
i64,i64,str,i64,str,str
56360,0,"""wt""",23,"""VUSDUALXP_23""","""A12"""
56360,0,"""wt""",23,"""VUSDUALXP_23""","""B12"""
56360,0,"""wt""",23,"""VUSDUALXP_23""","""C12"""
56360,0,"""wt""",23,"""VUSDUALXP_23""","""D12"""
9905,0,"""wt""",23,"""VUSDUALXP_23""","""F02"""
…,…,…,…,…,…
null,null,"""pdest_empty""",20,"""VUSDUALXP_20""","""A08"""
null,null,"""pdest_empty""",21,"""VUSDUALXP_21""","""A07"""
null,null,"""pdest_empty""",21,"""VUSDUALXP_21""","""A08"""


In [4]:
pdest_layout_df.filter((pl.col("dest_pla_id")==18)&(pl.col("dest_well")=="E11"))

orf_id,mut_id,node_type,dest_pla_id,dest_pla,dest_well
i64,i64,str,i64,str,str
3438,201180,"""allele""",18,"""VUSDUALXP_18""","""E11"""


## 2. Loading the sequence confirmed results

In [5]:
seq_confirm_code_df = pd.read_table(f"{allele_collection_inputs}/sequence_confirmation_class_code.tsv")
with pd.option_context('display.max_columns', None):
    display(seq_confirm_code_df)

,code,description
0,1,perfectly validated
1,2,"partially validated, >= 50% coverage"
2,3,wild type
3,4,"partial wild type, >= 50% coverage"
4,5,"target and off-target mutation, >= 50% coverage"
5,6,"off-target mutation, >= 50% coverage"
6,7,"truncated, < 50% coverage"
7,99,no reads


In [6]:
seq_confirm_res_df = pd.read_table(f"{allele_collection_outputs}/varchamp_seq_confirm_alleles_clinvar_gnomad_dbnsfp_struc_scores_OMIM_MOI_disease_modules.tsv", low_memory=False)
seq_confirm_res_df = pl.DataFrame(
    seq_confirm_res_df
).with_columns(
    pl.col("mut_id").cast(pl.Int64).alias("mut_id")
)
with pl.Config(set_tbl_cols=100):
    display(seq_confirm_res_df)

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,n2h_ref_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,dualip_ref_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class,mislocalization_ref_sequence_confirmation_class,gene_variant,chr_num,nuc_loc,ref_allele,alt_allele,Chrom,entry_seq_final,db_seq_final,n2h_seq_final,dualip_seq_final,mislocalization_seq_final,varid,#AlleleID,Type,…,GeneID_dbnsfp,GeneSymbol_dbnsfp,HGNC_ID_dbnsfp,ClinicalSignificance_dbnsfp,ClinSigSimple_dbnsfp,LastEvaluated_dbnsfp,RS# (dbSNP)_dbnsfp,nsv/esv (dbVar)_dbnsfp,RCVaccession_dbnsfp,PhenotypeIDS_dbnsfp,PhenotypeList_dbnsfp,Origin_dbnsfp,OriginSimple_dbnsfp,Assembly_dbnsfp,ChromosomeAccession_dbnsfp,Chromosome_dbnsfp,Start_dbnsfp,Stop_dbnsfp,Cytogenetic_dbnsfp,ReviewStatus_dbnsfp,NumberSubmitters_dbnsfp,Guidelines_dbnsfp,TestedInGTR_dbnsfp,OtherIDs_dbnsfp,SubmitterCategories_dbnsfp,VariationID_dbnsfp,PositionVCF_dbnsfp,ReferenceAlleleVCF_dbnsfp,AlternateAlleleVCF_dbnsfp,SomaticClinicalImpact_dbnsfp,SomaticClinicalImpactLastEvaluated_dbnsfp,ReviewStatusClinicalImpact_dbnsfp,Oncogenicity_dbnsfp,OncogenicityLastEvaluated_dbnsfp,ReviewStatusOncogenicity_dbnsfp,SCVsForAggregateGermlineClassification_dbnsfp,SCVsForAggregateSomaticClinicalImpact_dbnsfp,SCVsForAggregateOncogenicityClassification_dbnsfp,clinvar_nt_change_dbnsfp,clinvar_aa_change_dbnsfp,RefSeq_mRNA_dbnsfp,StarStatus_dbnsfp,clinvar_clnsig_clean_dbnsfp,plddt,rsa,OMIM_IDs,inheritance_pattern,disease_modules,disease_module_count,OMIM_disease_names
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,str,str,str,str,str,str,str,str,str,f64,str,…,f64,str,str,str,f64,str,f64,str,str,str,str,str,str,str,str,str,f64,f64,str,str,f64,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,f64,str,str,str,i64,str
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""",2.0,"""RC4_Mut_GDDh1026""","""H01""",null,null,null,null,null,null,1.0,1.0,1.0,2.0,0.0,null,null,0.0,null,null,0.0,null,null,"""GBA1_Val54Leu""",1.0,1.55240033e8,"""C""","""G""","""1""","""Y""","""Y""","""N""","""N""","""N""","""1-155240033-C-G""",null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""",2.0,"""RC4_Mut_GDDh1026""","""E01""",null,null,null,null,null,null,1.0,1.0,1.0,1.0,0.0,null,null,0.0,null,null,0.0,null,null,"""GBA1_Leu224Phe""",1.0,1.55238225e8,"""G""","""A""","""1""","""Y""","""Y""","""N""","""N""","""N""","""1-155238225-G-A""",null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null
"""GBA1""","""ENSG00000177628""",2,113,"""CCSBVarC000003""","""ALE00000003""","""NC_000001.11:155237453:C:T""","""887G>A""","""Arg296Gln""","""RC4""","""RC4_Mut_GDEh1026""","""F01""","""GDEhDisVCh_40

## 3. Map the pdest mutation to meta mutation information

In [7]:
# seq_confirm_res_df.filter(pl.col("mut_id")==0)

## map orf_id to symbol and ensembl_gene_id
orfid_2_symbol_mapping = dict(zip(
    seq_confirm_res_df.filter(pl.col("symbol").is_not_null())["orf_id"].to_list(),
    seq_confirm_res_df.filter(pl.col("symbol").is_not_null())["symbol"].to_list()
))

orfid_2_geneid_mapping = dict(zip(
    seq_confirm_res_df.filter(pl.col("ensembl_gene_id").is_not_null())["orf_id"].to_list(),
    seq_confirm_res_df.filter(pl.col("ensembl_gene_id").is_not_null())["ensembl_gene_id"].to_list()
))

In [8]:
## join the pdest variant with mut meta info by "INNER" JOIN;
## some alleles were filtered out during the allele collection stage due to their poor seq quality
## SHOULD REMOVE them from DUAL-IPA assay
pdest_layout_seq_conf_mut_df = pdest_layout_df.filter(
    pl.col("mut_id")!=0
).join(
    seq_confirm_res_df, 
    on=["orf_id", "mut_id"],
    how="inner"
)

## need to add wt data here as well
pdest_layout_seq_conf_wt_df = pdest_layout_df.filter(
    pl.col("mut_id")==0
).join(
    seq_confirm_res_df, 
    on=["orf_id", "mut_id"],
    how="left"
).with_columns(
    pl.col("aa_change").fill_null("WT"),
    pl.col("nt_change").fill_null("WT"),
    pl.col("orf_id").map_elements(lambda x: orfid_2_symbol_mapping[x] if x in orfid_2_symbol_mapping else pl.Null, return_dtype=pl.String).alias("symbol"),
    pl.col("orf_id").map_elements(lambda x: orfid_2_geneid_mapping[x] if x in orfid_2_geneid_mapping else pl.Null, return_dtype=pl.String).alias("ensembl_gene_id")
)

## need to add the pdest_empty wells as well
pdest_layout_seq_conf_empty_df = pdest_layout_df.filter(
    pl.col("mut_id").is_null()
).join(
    seq_confirm_res_df, 
    on=["orf_id", "mut_id"],
    how="left"
)

## concat var and wt and empty
pdest_layout_seq_conf_df = pl.concat([
    pdest_layout_seq_conf_mut_df, pdest_layout_seq_conf_wt_df, pdest_layout_seq_conf_empty_df
])
print(pdest_layout_seq_conf_df["dualip_sequence_confirmation_class"].unique())

shape: (9,)
Series: 'dualip_sequence_confirmation_class' [f64]
[
	null
	1.0
	2.0
	3.0
	4.0
	5.0
	6.0
	7.0
	99.0
]


In [9]:
pdest_layout_seq_conf_df.unique(subset="dualip_seq_final")["dualip_seq_final"]

dualip_seq_final
str
"""Y"""
"""N"""
null


In [10]:
pdest_layout_seq_conf_df.write_csv(f'{dualipa_outputs}/dualipa_experimental_layout.csv')